In [4]:
import pandas as pd
import numpy as np
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [8]:
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)


stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    words = text.split()
    cleaned_words = []
    for word in words:
        if word.startswith("http") or word.startswith("www") or word.startswith("@") or word.startswith("#"):
            continue
        word = word.strip(string.punctuation)
        if word.isdigit():
            continue
        if word in stop_words:
            continue
        if word:
            word = lemmatizer.lemmatize(word)
            cleaned_words.append(word)
    return " ".join(cleaned_words)


class MultinomialNaiveBayes:
    def __init__(self, nb_classes, nb_words, pseudocount=1):
        self.nb_classes = nb_classes
        self.nb_words = nb_words
        self.pseudocount = pseudocount

    def fit(self, X, Y):
        nb_examples = X.shape[0]
        self.priors = np.bincount(Y) / nb_examples
        print("Priors:")
        print(self.priors)

        occs = np.zeros((self.nb_classes, self.nb_words))
        for i in range(nb_examples):
            c = Y[i]
            for w in range(self.nb_words):
                occs[c][w] += X[i][w]

        self.like = np.zeros((self.nb_classes, self.nb_words))
        for c in range(self.nb_classes):
            for w in range(self.nb_words):
                up = occs[c][w] + self.pseudocount
                down = np.sum(occs[c]) + self.nb_words * self.pseudocount
                self.like[c][w] = up / down

    def predict(self, bow):
        probs = np.zeros(self.nb_classes)
        for c in range(self.nb_classes):
            prob = np.log(self.priors[c])
            for w in range(self.nb_words):
                cnt = bow[w]
                prob += cnt * np.log(self.like[c][w])
            probs[c] = prob
        prediction = np.argmax(probs)
        return prediction

    def predict_all(self, X):
        return np.array([self.predict(x) for x in X])


df = pd.read_csv("disaster-tweets.csv")
df = df.dropna(subset=["text", "target"])

df["text_clean"] = df["text"].apply(clean_text)

X_text = df["text_clean"].tolist()
Y = df["target"].values

vectorizer = CountVectorizer(max_features=5000)
X_bow = vectorizer.fit_transform(X_text).toarray()

accuracies = []

for run in range(1, 4):
    X_train, X_test, Y_train, Y_test = train_test_split(X_bow, Y, test_size=0.2, random_state=run)

    nb_classes = 2
    nb_words = X_train.shape[1]
    model = MultinomialNaiveBayes(nb_classes, nb_words, pseudocount=1)
    model.fit(X_train, Y_train)

    preds = model.predict_all(X_test)
    acc = accuracy_score(Y_test, preds)

    accuracies.append(acc)

avg_accuracy = np.mean(accuracies)
print(f"\n{'='*20} Rezultati nakon 3 pokretanja {'='*20}")
for i, acc in enumerate(accuracies, 1):
    print(f"Pokretanje {i}: {acc:.4f}")
print(f"Prosecna tacnost: {avg_accuracy:.4f}")


pos_indices = np.where(Y == 1)[0]
neg_indices = np.where(Y == 0)[0]

X_pos = X_bow[pos_indices]
X_neg = X_bow[neg_indices]


word_list = vectorizer.get_feature_names_out()

pos_counts = np.sum(X_pos, axis=0)
neg_counts = np.sum(X_neg, axis=0)


top_pos = sorted(zip(word_list, pos_counts), key=lambda x: x[1], reverse=True)[:5]
top_neg = sorted(zip(word_list, neg_counts), key=lambda x: x[1], reverse=True)[:5]

print("\n5 najcescih reci u pozitivnim tvitovima:")
for word, count in top_pos:
    print(f"{word}: {int(count)}")

print("\n5 najcescih reci u negativnim tvitovima:")
for word, count in top_neg:
    print(f"{word}: {int(count)}")

lr_words = []
for i, word in enumerate(word_list):
    if pos_counts[i] >= 10 and neg_counts[i] >= 10:
        lr = pos_counts[i] / neg_counts[i]
        lr_words.append((word, lr))


lr_sorted = sorted(lr_words, key=lambda x: x[1], reverse=True)
top_5_lr = lr_sorted[:5]
bottom_5_lr = lr_sorted[-5:]

print("\n5 reci sa najvecom vrednoscu LR metrike:")
for word, score in top_5_lr:
    print(f"{word}: {score:.2f}")

print("\n5 reci sa najmanjom vrednoscu LR metrike:")
for word, score in bottom_5_lr:
    print(f"{word}: {score:.2f}")





Priors:
[0.5681445 0.4318555]
Priors:
[0.57011494 0.42988506]
Priors:
[0.57487685 0.42512315]

==================== Rezultati nakon 3 pokretanja ====================
Pokretanje 1: 0.7978
Pokretanje 2: 0.7938
Pokretanje 3: 0.7938
Prosecna tacnost: 0.7951

5 najcescih reci u pozitivnim tvitovima:
fire: 265
amp: 135
via: 121
disaster: 114
suicide: 112

5 najcescih reci u negativnim tvitovima:
like: 255
amp: 208
get: 185
new: 170
one: 138

5 reci sa najvecom vrednoscu LR metrike:
train: 6.67
fatal: 4.58
family: 4.20
kill: 4.15
evacuation: 4.10

5 reci sa najmanjom vrednoscu LR metrike:
feel: 0.20
want: 0.17
let: 0.16
full: 0.12
love: 0.11


Analizom najcescih reci i LR metrika vidi se jasna razlika izmedju reci koriscenih u tvitovima o katastrofama i onim koje to nisu.
U pozitivnim tvitovima ( odnose se na katastrofe) najcesce se javljaju reci kao sto su: fire, disaster i evacuation koje vuku na nesrecu, dok se u negativnim (obicnim tvitovima) pojavljuju emotivnije reci kao sto su: love, feel i want.
LR metrika (likelihood ratio) se racuna kao odnos broja pojavljivanja reci u pozitivnim tvitovima naspram negativnih. Reci sa visokim LR (vise puta se javljaju u pozitivnim tvitovima)
vise su vezane za katastrofe, a reci sa niskim LR cesto nisu vezane za opis nesrece.
LR metrika nam pomaze da vidimo koje su reci specificne za pozitivne, a koje za negativne tvitove i to moze da poboljsa tacnost modela.